## 4.1 自动微分 - 核心概念

#### 1. 为什么需要自动微分：

##### 1.1 正规方程求解与梯度下降：
* ⚠️正规方程求解直接对损失函数求导并且使其等于0来一次性得到w，结果并不理想
* ✅梯度下降：
    * 首先通过设置的初始w和b，得到损失函数的结果
    * 将损失函数对w和b求偏导，得到中间结果 - 梯度
    * w新 = w旧 - 学习率*梯度

##### 1.2 自动微分
* 手动求导：对于包含数百万参数和复杂激活函数的深度神经网络，用纸笔推导解析解是不现实的，且极其容易出错。
* 数值求导：通过微小的变量扰动（如 hf(x+h)−f(x) ）来估算梯度，计算量极大，速度极慢。
* 自动微分（Auto-diff）：完美结合了前两者的优点，它通过代码在后台自动、精确且高效地计算出所有参数的梯度。

#### 2. 什么是 Autograd？

Autograd 是 PyTorch 内置的自动微分引擎：
* 动态计算图自动微分系统

可以把它理解为一个“无声的记录员”：
* 当你对张量（Tensor）进行各种数学运算（如加减乘除、矩阵点乘）时，Autograd 会在后台悄悄记录下所有的**操作轨迹**。
* 当需要反向传播更新参数时，它就能根据这份记录，瞬间算出所有需要的梯度。


#### 3. 什么是“计算图”（Computational Graph）？

**被记录下来的“操作轨迹”**在程序中被表示为一种数据结构：有向无环图（DAG），即计算图。
* 节点（Nodes）：图中的节点代表张量（Tensor）以及数学函数（Function）。
* 边（Edges）：代表数据的流向（输入输出关系）。

当你执行前向传播（Forward Pass）时，PyTorch 会动态构建这张图，记录输入数据是如何一步步变成最终的 Loss 的。


In [1]:
import torch
x = torch.tensor(2.0, requires_grad=True)
y = x * x
z = y + 3

内部结构是：

Node：x <br>
    │ <br>
    * (x*x) <br>
    │ <br>
Node：y <br>
    │ <br>
    +3 <br>
    │ <br>
Node：z

#### 4. requires_grad（最关键参数）

这是决定一个张量是否被加入计算图的“通行证”。
* 告诉 PyTorch：这个变量需要计算梯度
* 如果不写,`requires_grad=False`,系统不会构建计算图。

#### 5. 反向传播：backward() 

当计算出最终的标量误差（如 loss）后，我们只需调用一行代码：loss.backward()：
* 这个方法会触发引擎沿着刚才建立好的“计算图”反向回溯，
* 自动计算出 loss 对图中所有 requires_grad=True 的张量的梯度，
* 并将结果存储在各个张量的 .grad 属性中（例如 weight.grad）。

In [2]:
import torch
x = torch.tensor(2.0, requires_grad=True)
y = x * x
z = y + 3

z.backward()

print(x.grad)

tensor(4.)


#### 6. 自动微分的本质原理（必须理解）

Autograd 使用的是：链式法则（Chain Rule）

##### 6.1 什么是链式法则：
1. 是微积分中求导数的一个核心法则，专门用于处理 复合函数（Composite Function）的求导。
2. 简单来说，当一个函数是由两个或多个函数“嵌套”在一起组成时（比如“洋葱”一样一层包一层），你就需要用链式法则来求它的导数。
    * 外层函数（Outside Function）：最外面的运算。
    * 内层函数（Inside Function）：被包裹在里面的运算。

##### 6.2 链式法则的核心思想：
1. 先对外层求导（保持内层不变），然后乘以对内层的求导。即“由外向内，层层求导，最后相乘”。
2. 如果函数 y=f(g(x))，其中 u=g(x) 是内层函数，y=f(u) 是外层函数，那么对 x 的导数：
`d(y)/d(x) = d(y)/d(u) * d(u)/d(x)`

#### 7. 动态图 vs 静态图⚠️

动态图 (Dynamic Graph / Define-by-Run)：
* 计算图是在代码运行的过程中实时构建的。
* 每一次前向传播都会生成一张全新的图。
* 这使得使用 Python 的 if/else、循环等控制流变得非常容易，调试（Debug）时也可以直接打印变量。

静态图 (Static Graph / Define-and-Run)：
* 需要先像写图纸一样定义好整个计算逻辑，
* 然后再把数据“喂”进去执行
* 执行速度稍快，但缺乏灵活性，调试困难。

#### 8. 叶子节点（Leaf Tensor）概念 🌿

什么是叶子节点？:
* 简单来说，直接由用户创建，且不是由其他运算生成的张量就是叶子节点。
* `x = torch.tensor(2.0, requires_grad=True)` 此时 x 就是叶子节点

规则：
* 在整张计算图中，只有叶子节点的梯度会被永久保留下来（为了节省内存）。

#### 9. 梯度默认会累加 ⚠️

如果你连续多次调用 loss.backward()，新计算出来的梯度不会覆盖原来的值，而是会累加（Add）到张量原有的 .grad 中。

为什么要这么设计？ ：
* 在显存有限时，我们可以利用这个特性，将一个大的 Batch 分成几个小的 Mini-batch 输入，多次反向传播累加梯度后，再执行一次参数更新，从而变相实现大 Batch 训练。

必须要做的事：
* 每次准备进行新一轮参数更新前，必须手动清零梯度，调用 optimizer.zero_grad()。

In [4]:
a = torch.tensor(2.0, requires_grad=True)

b = a * a 
b.backward()

b = a * a
b.backward()
print(a.grad)

tensor(8.)
